In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_train = pd.read_csv('data/ratings_train.txt', delimiter='\t', keep_default_na=False)
df_train.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [3]:
text_train, y_train = df_train['document'].values, df_train['label'].values

In [4]:
df_test = pd.read_csv('data/ratings_test.txt', delimiter='\t', keep_default_na=False)
text_test, y_test = df_test['document'].values, df_test['label'].values

In [5]:
from konlpy.tag import Okt

class PicklableOkt(Okt):
    
    def __init__(self, *args):
        self.args = args
        Okt.__init__(self, *args)
        
    def __setstate__(self, state):
        self.__init__(*state['args'])
        
    def __getstate__(self):
        return {'args':self.args}

okt = PicklableOkt()

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV

param_grid = {'tfidfvectorizer__min_df':[3,5,7],
              'tfidfvectorizer__ngram_range':[(1,1),(1,2),(1,3)], 
              'logisticregression__C':[0,1,1,10]}
pipe = make_pipeline(TfidfVectorizer(tokenizer=okt.morphs),
                     LogisticRegression())
grid = GridSearchCV(pipe, param_grid, n_jobs=-1)

# 그리드 서치를 수행합니다
grid.fit(text_train[:1000], y_train[:1000])
print('최상의 교차 검증 점수: {:.3f}'.format(grid.best_score_))
print('최적의 교차 검증 매개변수:', grid.best_params_)

최상의 교차 검증 점수: 0.718
최적의 교차 검증 매개변수: {'logisticregression__C': 1, 'tfidfvectorizer__min_df': 3, 'tfidfvectorizer__ngram_range': (1, 3)}


In [7]:
tfidfvectorizer = grid.best_estimator_.named_steps['tfidfvectorizer']
x_test = tfidfvectorizer.transform(text_test[:1000])
logisticregression = grid.best_estimator_.named_steps['logisticregression']
score = logisticregression.score(x_test, y_test[:1000])
print('테스트 세트 점수: {:.3f}'.format(score))

테스트 세트 점수: 0.714


In [8]:
from konlpy.tag import Mecab

class PicklableMecab(Mecab):
    
    def __init__(self, *args):
        self.args = args
        Mecab.__init__(self, *args)
        
    def __setstate__(self, state):
        self.__init__(*state['args'])
        
    def __getstate__(self):
        return {'args':self.args}
    
mecab = PicklableMecab()

Exception: Install MeCab in order to use it: http://konlpy.org/en/latest/install/